# CS2 Demo Analizi - awpy ile Veri Çekme
Bu notebook, `.dem` dosyasından tüm oyun verilerini çeker ve `demo_data.json` olarak export eder.

In [ ]:
# Gerekli kütüphaneleri kur
import subprocess
subprocess.run(['pip', 'install', 'awpy', 'pandas'], capture_output=False)

# Notebook genelinde isimler tanımlı kalsın (lint/analysis için)
DEMO_FILE = 'vita-auro.dem'
dem = None

In [ ]:
from awpy import Demo
import json
import math

# ─── Demo dosyasını yükle ──────────────────────────────────────────────────────
DEMO_FILE = "vita-auro.dem"

print(f"Demo yükleniyor: {DEMO_FILE} ...")
dem = Demo(DEMO_FILE, verbose=True)

base_props = [
    "X", "Y", "Z",
    "health",
    "armor_value",
    "has_helmet",
    "has_defuser",
    "inventory",
    "current_equip_value",
    "team_name",
    "is_alive",
    "pitch",
    "yaw",
    "active_weapon",
]

extra_props = ["cash", "has_bomb"]

try:
    dem.parse(player_props=base_props + extra_props)
except Exception as ex:
    print("⚠️  Ek player_props parse edilemedi, temel alanlarla devam ediliyor.")
    print(f"Sebep: {ex}")
    dem = Demo(DEMO_FILE, verbose=True)
    dem.parse(player_props=base_props)

print("\n✅ Parse tamamlandı!")
print(f"📍 Harita : {dem.header.get('map_name', 'Bilinmiyor')}")
print(f"🖥️  Sunucu : {dem.header.get('server_name', 'Bilinmiyor')}")
print(f"🎮 Demo   : {dem.header.get('demo_version_name', 'Bilinmiyor')}")
print(f"\nHeader bilgileri:")
for k, v in dem.header.items():
    print(f"  {k}: {v}")

Demo yükleniyor: vita-auro.dem ...
2026-04-17 14:40:00.188 | DEBUG    | awpy.demo:parse:220 - Starting to parse vita-auro.dem
2026-04-17 14:40:21.422 | SUCCESS  | awpy.demo:parse:271 - Finished parsing vita-auro.dem, took 21.23 seconds

✅ Parse tamamlandı!
📍 Harita : de_inferno
🖥️  Sunucu : BLAST.tv Premier CS2 Server
🎮 Demo   : valve_demo_2

Header bilgileri:
  allow_clientside_particles: True
  demo_version_name: valve_demo_2
  map_name: de_inferno
  addons: 
  client_name: SourceTV Demo
  game_directory: /home/steam/cs2/game/csgo
  server_name: BLAST.tv Premier CS2 Server
  allow_clientside_entities: True
  demo_file_stamp: PBDEMS2 
  patch_version: 14140
  fullpackets_version: 2
  demo_version_guid: 8e9d71ab-04a1-4c01-bb61-acfede27c046


In [ ]:
# ─── Veri Boyutlarına Genel Bakış ─────────────────────────────────────────────
print("=" * 55)
print("📊  VERİ TABLOSU BOYUTLARI")
print("=" * 55)

tables = {
    "rounds"    : dem.rounds,
    "ticks"     : dem.ticks,
    "kills"     : dem.kills,
    "damages"   : dem.damages,
    "shots"     : dem.shots,
    "bomb"      : dem.bomb,
    "grenades"  : dem.grenades,
    "smokes"    : dem.smokes,
    "infernos"  : dem.infernos,
    "footsteps" : dem.footsteps,
}

for name, df in tables.items():
    if df is not None and len(df) > 0:
        print(f"  {name:<12}: {df.shape[0]:>7,} satır  x  {df.shape[1]:>3} sütun")
    else:
        print(f"  {name:<12}: (boş)")

print("=" * 55)

In [ ]:
import polars as pl

# ─── Kill İstatistikleri ───────────────────────────────────────────────────────
print("=" * 60)
print("💀  KILL TABLOSU")
print("=" * 60)

kills_df = dem.kills

# Attacker kill sayısı
kill_counts = (
    kills_df
    .filter(pl.col("attacker_name").is_not_null())
    .group_by("attacker_name", "attacker_team_name")
    .agg(pl.len().alias("kills"))
    .sort("kills", descending=True)
)

# Death sayısı
death_counts = (
    kills_df
    .filter(pl.col("victim_name").is_not_null())
    .group_by("victim_name")
    .agg(pl.len().alias("deaths"))
)

# Join
kd_table = kill_counts.join(
    death_counts,
    left_on="attacker_name",
    right_on="victim_name",
    how="left"
).with_columns(
    (pl.col("kills") / pl.col("deaths").fill_null(1)).round(2).alias("K/D")
).sort("kills", descending=True)

print(kd_table)

print("\n📌 En çok kullanılan silahlar:")
print(
    kills_df
    .filter(pl.col("weapon").is_not_null())
    .group_by("weapon")
    .agg(pl.len().alias("kill_sayisi"))
    .sort("kill_sayisi", descending=True)
    .head(10)
)

In [ ]:
# ─── Round Özeti ──────────────────────────────────────────────────────────────
print("=" * 60)
print("🔄  ROUND TABLOSU")
print("=" * 60)
print(dem.rounds)

# Skor hesapla
rounds_pd = dem.rounds.to_pandas()
ct_wins = (rounds_pd["winner"] == "CT").sum()
t_wins  = (rounds_pd["winner"] == "T").sum()
total   = len(rounds_pd)

print(f"\n🏆 Skor: CT {ct_wins} - {t_wins} T  (toplam {total} round)")

# Bomba istatistikleri
planted  = rounds_pd["bomb_plant"].notna().sum()
exploded = (rounds_pd["reason"] == "bomb_exploded").sum()
defused  = (rounds_pd["reason"] == "bomb_defused").sum()

print(f"💣 Bomba Ekildi: {planted} | Patladı: {exploded} | Etkisiz: {defused}")

In [ ]:
# ─── Bomba Olayları ───────────────────────────────────────────────────────────
print("=" * 60)
print("💣  BOMBA OLAYLARI")
print("=" * 60)
print(dem.bomb)

print("\n📍 Bomba yerleştirme konumları (plant olayları):")
bomb_plants = dem.bomb.filter(pl.col("event") == "plant")
if len(bomb_plants) > 0:
    print(bomb_plants.select(["tick", "name", "X", "Y", "Z", "bombsite"]))
else:
    print("  Bu demoda bomba yerleştirilmemiş.")

In [ ]:
# ─── Oyuncu Pozisyonlarına Bakış ──────────────────────────────────────────────
print("=" * 60)
print("📍  TİCK VERİSİ (İlk 5 Satır)")
print("=" * 60)
print(dem.ticks.head(5))

print("\nSütunlar:")
print(dem.ticks.columns)

# Oyuncu listesi
players = dem.ticks.select(["name", "steamid", "team_name"]).unique(subset="steamid")
print("\n👥 Oyuncular:")
print(players)

In [ ]:
# ─── JSON Export (Replay Tool için) ───────────────────────────────────────────
import json
import math
import polars as pl
from pathlib import Path

print("📦 JSON export başlıyor...")

def safe_val(v):
    """NaN/Inf → None, diğerleri olduğu gibi"""
    if v is None:
        return None
    if isinstance(v, float) and (math.isnan(v) or math.isinf(v)):
        return None
    if isinstance(v, list):
        return [safe_val(x) for x in v]
    if isinstance(v, dict):
        return {k: safe_val(x) for k, x in v.items()}
    return v

def df_to_list(df, sample_every=1):
    """Polars DataFrame → Python list of dicts"""
    if df is None or len(df) == 0:
        return []
    if sample_every > 1:
        df = df[::sample_every]
    rows = df.to_dicts()
    cleaned = []
    for row in rows:
        cleaned.append({k: safe_val(v) for k, v in row.items()})
    return cleaned

# Tick verisi büyük olabilir → her 4 tickte bir al
TICK_SAMPLE = 4
ticks_sampled = dem.ticks[::TICK_SAMPLE]
print(f"  Ticks: {len(dem.ticks):,} → örnekleme sonrası {len(ticks_sampled):,} satır (her {TICK_SAMPLE}. tick)")

# Grenade trajektorylerini de örnekle
grenades_sampled = dem.grenades[::2] if dem.grenades is not None and len(dem.grenades) else None

# Harita sınırları (render için world → canvas normalize)
map_bounds = None
if dem.ticks is not None and len(dem.ticks):
    xs = [x for x in dem.ticks["X"].to_list() if x is not None and not (isinstance(x, float) and (math.isnan(x) or math.isinf(x)))]
    ys = [y for y in dem.ticks["Y"].to_list() if y is not None and not (isinstance(y, float) and (math.isnan(y) or math.isinf(y)))]
    if xs and ys:
        pad = 200
        map_bounds = {
            "min_x": min(xs) - pad,
            "max_x": max(xs) + pad,
            "min_y": min(ys) - pad,
            "max_y": max(ys) + pad,
        }

# Otomatik Harita Ray Tespiti
map_name = dem.header.get('map_name', 'unknown')
map_image_path = None
if map_name and map_name != 'unknown':
    map_file = Path('maps') / f"{map_name}.png"
    if map_file.exists():
        map_image_path = f"maps/{map_name}.png"
        print(f"  🗺️  Map resmi bulundu: {map_image_path}")

# Bomba taşıyan oyuncu rotası
bomb_carrier_path = []
if dem.ticks is not None and len(dem.ticks) and "has_bomb" in dem.ticks.columns:
    bomb_carrier_path = (
        dem.ticks
        .filter(pl.col("has_bomb") == True)
        .select([c for c in ["round_num", "tick", "steamid", "name", "X", "Y", "Z", "has_bomb"] if c in dem.ticks.columns])
        .sort(["round_num", "tick"])
        .to_dicts()
    )

# Grenade iniş/target noktaları
grenade_landings = []
if dem.grenades is not None and len(dem.grenades):
    req = {"round_num", "entity_id", "tick", "X", "Y", "Z"}
    if req.issubset(set(dem.grenades.columns)):
        g = dem.grenades.sort("tick")
        g_start = g.group_by(["round_num", "entity_id"]).first()
        g_end = g.group_by(["round_num", "entity_id"]).last()

        start_cols = [c for c in ["round_num", "entity_id", "tick", "thrower_name", "thrower_steamid", "grenade_type"] if c in g_start.columns]
        end_cols = [c for c in ["round_num", "entity_id", "tick", "X", "Y", "Z"] if c in g_end.columns]

        g_start = g_start.select(start_cols).rename({"tick": "start_tick"})
        g_end = g_end.select(end_cols).rename({"tick": "land_tick", "X": "land_X", "Y": "land_Y", "Z": "land_Z"})

        grenade_landings = g_start.join(g_end, on=["round_num", "entity_id"], how="inner").to_dicts()

# Oyuncu kill sayıları
player_kills = []
if dem.kills is not None and len(dem.kills) and "attacker_steamid" in dem.kills.columns:
    grp_cols = [c for c in ["attacker_steamid", "attacker_name", "attacker_team_name"] if c in dem.kills.columns]
    player_kills = (
        dem.kills
        .filter(pl.col("attacker_steamid").is_not_null())
        .group_by(grp_cols)
        .agg(pl.len().alias("kills"))
        .sort("kills", descending=True)
        .to_dicts()
    )

demo_data = {
    "meta": {
        "map_name"    : map_name,
        "server_name" : dem.header.get("server_name", "unknown"),
        "demo_file"   : DEMO_FILE,
        "demo_version_name": dem.header.get("demo_version_name", "unknown"),
        "tick_sample" : TICK_SAMPLE,
        "map_bounds"  : map_bounds,
        "map_image"   : map_image_path,
    },
    "rounds"   : df_to_list(dem.rounds),
    "ticks"    : df_to_list(ticks_sampled),
    "kills"    : df_to_list(dem.kills),
    "damages"  : df_to_list(dem.damages),
    "bomb"     : df_to_list(dem.bomb),
    "grenades" : df_to_list(grenades_sampled),
    "smokes"   : df_to_list(dem.smokes),
    "infernos" : df_to_list(dem.infernos),
    "shots"    : df_to_list(dem.shots),
    "bomb_carrier_path": df_to_list(pl.DataFrame(bomb_carrier_path)) if len(bomb_carrier_path) else [],
    "grenade_landings": df_to_list(pl.DataFrame(grenade_landings)) if len(grenade_landings) else [],
    "player_kills": df_to_list(pl.DataFrame(player_kills)) if len(player_kills) else [],
}

OUT_FILE = "demo_data.json"
with open(OUT_FILE, "w", encoding="utf-8") as f:
    json.dump(demo_data, f, ensure_ascii=False, default=str)

import os
size_mb = os.path.getsize(OUT_FILE) / 1024 / 1024
print(f"✅ Export tamamlandı → {OUT_FILE}  ({size_mb:.1f} MB)")
print(f"🗺️  Harita: {demo_data['meta']['map_name']}")
print(f"📊 Tablo sayıları:")
for k in [
    "rounds", "ticks", "kills", "damages", "bomb", "grenades", "smokes", "infernos", "shots",
    "bomb_carrier_path", "grenade_landings", "player_kills",
]:
    print(f"   {k:<17}: {len(demo_data[k]):>8,} kayıt")

📦 JSON export başlıyor...
  Ticks: 1,356,570 → örnekleme sonrası 339,143 satır (her 4. tick)
  🗺️  Map resmi bulundu: maps/de_inferno.png
✅ Export tamamlandı → demo_data.json  (318.6 MB)
🗺️  Harita: de_inferno
📊 Tablo sayıları:
   rounds           :       18 kayıt
   ticks            :  339,143 kayıt
   kills            :      114 kayıt
   damages          :      563 kayıt
   bomb             :      101 kayıt
   grenades         : 1,001,952 kayıt
   smokes           :      108 kayıt
   infernos         :       99 kayıt
   shots            :    3,154 kayıt
   bomb_carrier_path:        0 kayıt
   grenade_landings :      907 kayıt
   player_kills     :       10 kayıt
